In [1]:
import sys
import pandas as pd
import re
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2_contingency
import numpy as np
from statsmodels.stats.contingency_tables import Table2x2
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
print(sys.executable)

/Users/ahthini/Desktop/DissProject/my_env/bin/python


### **T2.2 - Data cleaning and preprocessing**
#### **T2.2.1 - Clean admission-level modelling dataset**

**Load the initial extracted dataset**

The dataset created in T2.1 is loaded as the starting point for data cleaning and preprocessing. This dataset contains the psychiatric readmission outcome together with demographic, admission-level, ICU, diagnosis, medication, and laboratory features. Before modelling, the dataset must be checked for missing values, duplicate rows, implausible values, possible data leakage, and consistent feature formatting.

In [2]:
#set output folder where 2.1 was saved and load initial modelling dataset
output_path = Path("/Users/ahthini/Desktop/DissProject/outputs")
input_file = output_path / "t2_1_initial_modelling_dataset.csv"
df = pd.read_csv(input_file)

#convert admission and discharge timestamps to datetime format
for col in ["admittime", "dischtime"]:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors="coerce")

print("Loaded dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())
print("\nPreview:")
print(df.head())

Loaded dataset shape: (238565, 759)

Columns:
['subject_id', 'hadm_id', 'admittime', 'dischtime', 'days_to_next', 'readmitted_30d', 'gender', 'anchor_age', 'anchor_year_group', 'admission_type', 'admission_location', 'discharge_location', 'insurance', 'language', 'marital_status', 'race', 'hospital_los_days', 'discharged_against_advice', 'not_married_flag', 'single_or_divorced_or_widowed', 'non_english_language_flag', 'public_insurance_flag', 'medicaid_flag', 'medicare_flag', 'insurance_missing_flag', 'admitted_from_facility_flag', 'admitted_from_hospital_transfer_flag', 'emergency_room_admission_flag', 'transfer_from_hospital_flag', 'transfer_from_snf_flag', 'internal_transfer_from_psych_flag', 'discharged_home_flag', 'discharged_to_facility_flag', 'discharged_to_psych_facility_flag', 'los_under_2_days', 'los_under_7_days', 'los_7_to_30_days', 'los_30plus_days', 'los_60plus_days', 'previous_total_admissions', 'previous_psych_admissions_from_all_hosp', 'previous_nonpsych_admissions', '

**Assess hospital length of stay values**

Hospital length of stay was examined for implausible values prior to cleaning. Negative LOS values are not clinically meaningful and may indicate timestamp recording inconsistencies. Any admissions with negative LOS values will be investigated and removed if confirmed to represent data quality anomalies.

In [3]:
negative_los = df[df["hospital_los_days"] < 0]
print("Negative LOS rows:", len(negative_los))
print(negative_los[["subject_id", "hadm_id", "admittime", "dischtime", "hospital_los_days"]].head(20))

Negative LOS rows: 74
       subject_id   hadm_id           admittime           dischtime  \
908      10035271  26463092 2165-08-13 14:00:00 2165-08-13 00:00:00   
3200     10136283  29334858 2163-03-01 20:43:00 2163-03-01 02:32:00   
11437    10504539  21379384 2131-05-28 22:22:00 2131-05-28 17:25:00   
12086    10531790  28680274 2182-01-03 02:44:00 2182-01-03 00:00:00   
12748    10558628  28309753 2120-04-06 03:15:00 2120-04-06 02:53:00   
12869    10564068  22025371 2118-05-23 04:59:00 2118-05-23 00:00:00   
18082    10771869  21056293 2188-12-03 14:22:00 2188-12-03 00:00:00   
22195    10946397  26365751 2127-03-24 07:19:00 2127-03-24 07:08:00   
30877    11314326  20771794 2140-09-05 02:53:00 2140-09-05 02:40:00   
33008    11401579  29426063 2127-09-10 01:59:00 2127-09-10 01:30:00   
33882    11436324  24532041 2110-02-14 02:13:00 2110-02-14 00:00:00   
42746    11818101  27807622 2163-08-20 01:04:00 2163-08-20 01:03:00   
46660    11976503  21590700 2165-07-24 05:05:00 2165-07

**Initial data quality checks**

The dataset is first checked for duplicate admission records, missing values, and outcome label distribution. Since the dataset is admission-level, each `subject_id` and `hadm_id` combination should appear only once.

In [4]:
print("Dataset shape:", df.shape)

#check whether any admissions appear more than once
print("\nDuplicate subject_id + hadm_id rows:")
print(df.duplicated(subset=["subject_id", "hadm_id"]).sum())

#check distribution of target variable before cleaning
print("\nReadmission label distribution:")
print(df["readmitted_30d"].value_counts())

#check % distribution of target variable
print("\nReadmission label distribution (%):")
print(df["readmitted_30d"].value_counts(normalize=True) * 100)

#create missing value summary for all columns
missing_summary = pd.DataFrame({"missing_count": df.isna().sum(),
    "missing_percent": (df.isna().mean() * 100).round(2)
}).sort_values("missing_percent", ascending=False)

print("\nMissing values summary:")
print(missing_summary)

Dataset shape: (238565, 759)

Duplicate subject_id + hadm_id rows:
0

Readmission label distribution:
readmitted_30d
0    191198
1     47367
Name: count, dtype: int64

Readmission label distribution (%):
readmitted_30d
0    80.145034
1    19.854966
Name: proportion, dtype: float64

Missing values summary:
                                         missing_count  missing_percent
first_discharge_planning_order_hours            238565           100.00
last_discharge_planning_order_hours             238565           100.00
ed_to_ward_delay_hours                          235025            98.52
systolic_bp_arterial_vital_value_change         224009            93.90
systolic_bp_arterial_vital_first_value          224009            93.90
...                                                ...              ...
num_stopped_procedureevents                          0             0.00
icu_event_interruption_score                         0             0.00
icu_event_density_per_day                    

**Remove leakage variable**

The variable `days_to_next` was useful when defining the 30-day readmission outcome in T1.4. However, it shouldn't be used as a predictor as it contains future information about when the next admission occurs. A clinician would not know this value at the time of discharge. Therefore, it is removed before modelling.

In [5]:
#remove days_to_next as it contains info from future admissions and was only used during outcome creation
if "days_to_next" in df.columns:
    df = df.drop(columns=["days_to_next"])

print("Shape after removing leakage variable:", df.shape)
print("Columns after removal:")
print(df.columns.tolist())

Shape after removing leakage variable: (238565, 758)
Columns after removal:
['subject_id', 'hadm_id', 'admittime', 'dischtime', 'readmitted_30d', 'gender', 'anchor_age', 'anchor_year_group', 'admission_type', 'admission_location', 'discharge_location', 'insurance', 'language', 'marital_status', 'race', 'hospital_los_days', 'discharged_against_advice', 'not_married_flag', 'single_or_divorced_or_widowed', 'non_english_language_flag', 'public_insurance_flag', 'medicaid_flag', 'medicare_flag', 'insurance_missing_flag', 'admitted_from_facility_flag', 'admitted_from_hospital_transfer_flag', 'emergency_room_admission_flag', 'transfer_from_hospital_flag', 'transfer_from_snf_flag', 'internal_transfer_from_psych_flag', 'discharged_home_flag', 'discharged_to_facility_flag', 'discharged_to_psych_facility_flag', 'los_under_2_days', 'los_under_7_days', 'los_7_to_30_days', 'los_30plus_days', 'los_60plus_days', 'previous_total_admissions', 'previous_psych_admissions_from_all_hosp', 'previous_nonpsych_

**Assess hospital length of stay values**

Hospital length of stay was calculated as the difference between discharge time and admission time. Negative length of stay values are not meaningful and may indicate timestamp recording inconsistencies. These rows are inspected before removal.

In [6]:
negative_los = df[df["hospital_los_days"] < 0] #admissions with -ve LOS

print("Negative LOS rows:", len(negative_los)) #no. of affected admissions
print("\nExample negative LOS rows:") #example records for inspection
print(negative_los[["subject_id", "hadm_id", "admittime", "dischtime", "hospital_los_days"]].head(10))
print("\nPercentage of dataset with negative LOS:")
print((len(negative_los) / len(df)) * 100)

Negative LOS rows: 74

Example negative LOS rows:
       subject_id   hadm_id           admittime           dischtime  \
908      10035271  26463092 2165-08-13 14:00:00 2165-08-13 00:00:00   
3200     10136283  29334858 2163-03-01 20:43:00 2163-03-01 02:32:00   
11437    10504539  21379384 2131-05-28 22:22:00 2131-05-28 17:25:00   
12086    10531790  28680274 2182-01-03 02:44:00 2182-01-03 00:00:00   
12748    10558628  28309753 2120-04-06 03:15:00 2120-04-06 02:53:00   
12869    10564068  22025371 2118-05-23 04:59:00 2118-05-23 00:00:00   
18082    10771869  21056293 2188-12-03 14:22:00 2188-12-03 00:00:00   
22195    10946397  26365751 2127-03-24 07:19:00 2127-03-24 07:08:00   
30877    11314326  20771794 2140-09-05 02:53:00 2140-09-05 02:40:00   
33008    11401579  29426063 2127-09-10 01:59:00 2127-09-10 01:30:00   

       hospital_los_days  
908            -0.583333  
3200           -0.757639  
11437          -0.206250  
12086          -0.113889  
12748          -0.015278  
12869 

**Remove admissions with negative hospital length of stay**

A small number of admissions were found to have negative hospital length of stay values. These values likely reflect timestamp inconsistencies rather than true clinical admissions. As they represent a very small proportion of the dataset, these rows are removed.

In [7]:
#store dataset before removing invalid LOS records
df_before_los_removal = df.copy()

#identify admissions with negative hospital LOS
negative_los = df_before_los_removal[df_before_los_removal["hospital_los_days"] < 0]

print("Rows before removing negative LOS:", len(df_before_los_removal))
print("Negative LOS rows:", len(negative_los))

print("\nReadmission distribution among removed negative LOS rows:")
print(negative_los["readmitted_30d"].value_counts(dropna=False))

print("\nReadmission distribution among removed negative LOS rows (%):")
print((negative_los["readmitted_30d"].value_counts(normalize=True, dropna=False) * 100).round(2))

#remove admissions with negative LOS
df = df[df["hospital_los_days"] >= 0].copy()

print("\nRows after removing negative LOS:", len(df))
print("Rows removed:", len(df_before_los_removal) - len(df))

print("\nReadmission distribution after removal:")
print(df["readmitted_30d"].value_counts())

print("\nReadmission distribution after removal (%):")
print((df["readmitted_30d"].value_counts(normalize=True) * 100).round(2))

print("\nRemaining negative LOS rows:")
print((df["hospital_los_days"] < 0).sum())

Rows before removing negative LOS: 238565
Negative LOS rows: 74

Readmission distribution among removed negative LOS rows:
readmitted_30d
0    65
1     9
Name: count, dtype: int64

Readmission distribution among removed negative LOS rows (%):
readmitted_30d
0    87.84
1    12.16
Name: proportion, dtype: float64

Rows after removing negative LOS: 238491
Rows removed: 74

Readmission distribution after removal:
readmitted_30d
0    191133
1     47358
Name: count, dtype: int64

Readmission distribution after removal (%):
readmitted_30d
0    80.14
1    19.86
Name: proportion, dtype: float64

Remaining negative LOS rows:
0


**Handle missing values across expanded EHR feature domains**

Missing values were handled according to feature type. Administrative categorical variables were assigned an "Unknown" category, ICU-related categorical variables were assigned "No ICU", and service variables were assigned "No service record". Count and indicator variables were filled with 0 where absence of a record indicated no observed exposure or event. Continuous laboratory and vital sign values were median-imputed after creating missingness indicators, so that measurement availability was retained as a predictor.

In [8]:
#identify expanded feature groups from the T2.1 dataset, general categorical variables with administrative missingness
general_categorical_missing_cols = ["discharge_location", "insurance", "language",  "marital_status"]
for col in general_categorical_missing_cols:
    if col in df.columns:
        df[col] = df[col].fillna("Unknown")

#ICU categorical variables
icu_categorical_cols = ["first_icu_careunit", "last_icu_careunit", "icu_type"]
for col in icu_categorical_cols:
    if col in df.columns:
        df[col] = df[col].fillna("No ICU")

#service categorical variables
service_categorical_cols = ["first_service", "last_service"]
for col in service_categorical_cols:
    if col in df.columns:
        df[col] = df[col].fillna("No service record")

#diagnosis, medication, ICU, service, DRG, lab, and vital feature groups
diagnosis_cols = [col for col in df.columns if col.startswith("num_") and "diagnos" in col
    or col.startswith("has_")]

medication_cols = ["num_prescription_rows", "num_unique_drugs", "num_psych_med_classes",
    "had_antidepressant", "had_antipsychotic", "had_mood_stabiliser", "had_benzodiazepine",
    "had_stimulant", "had_sedative_hypnotic"]

prior_admission_cols = ["previous_total_admissions", "previous_psych_admissions_from_all_hosp", "previous_nonpsych_admissions"]
icu_numeric_cols = ["icu_stay_count", "total_icu_los_days", "had_icu_stay"]
service_numeric_cols = ["num_service_records",  "num_unique_services", "num_service_transfers", "had_service_transfer"]
physical_transfer_numeric_cols = ["num_transfer_events", "num_careunit_transfers",
    "num_unique_careunits", "had_ed_transfer_record", "had_unknown_transfer_careunit"]

procedure_numeric_cols = ["num_procedures", "num_unique_procedure_codes", "had_procedure"]
drg_numeric_cols = ["drg_severity", "drg_mortality", "drg_code_count"]

lab_indicator_cols = [col for col in df.columns
    if "_lab_" in col and col.endswith(("_lab_measured", "_lab_abnormal", "_lab_severe_abnormal",
        "_lab_count", "_lab_first_24h_count", "_lab_first_72h_count"))]

lab_value_cols = [col for col in df.columns if "_lab_" in col and (col.endswith("_first_value")
        or col.endswith("_last_value") or col.endswith("_mean_value") or col.endswith("_min_value")
        or col.endswith("_max_value") or col.endswith("_std_value") or col.endswith("_value_change")
        or col.endswith("_value_range") or col.endswith("_first_24h_mean")
        or col.endswith("_first_24h_min") or col.endswith("_first_24h_max")
        or col.endswith("_first_72h_mean") or col.endswith("_first_72h_min")
        or col.endswith("_first_72h_max"))]

vital_indicator_cols = [col for col in df.columns if col.endswith("_vital_measured")]
vital_value_cols = [col for col in df.columns if "_vital_" in col and not col.endswith("_vital_measured")]

#fill count and indicator columns with 0
zero_fill_cols = (diagnosis_cols + medication_cols + prior_admission_cols + icu_numeric_cols
    + service_numeric_cols + physical_transfer_numeric_cols + procedure_numeric_cols
    + drg_numeric_cols + lab_indicator_cols + vital_indicator_cols)

zero_fill_cols = [col for col in zero_fill_cols if col in df.columns]
for col in zero_fill_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

#DRG code count of 0 indicates no APR-DRG severity/mortality record
if "drg_code_count" in df.columns:
    df["has_drg_record"] = (df["drg_code_count"] > 0).astype(int)

#convert suitable count/indicator columns to integer
integer_cols = [col for col in zero_fill_cols if col != "total_icu_los_days"]

for col in integer_cols:
    df[col] = df[col].astype(int)

#add missingness indicators before imputing continuous lab and vital values
continuous_measurement_cols = lab_value_cols + vital_value_cols
continuous_measurement_cols = [col for col in continuous_measurement_cols if col in df.columns]

#create all missingness indicators at once to avoid dataframe fragmentation warnings
missing_indicator_df = pd.DataFrame({f"{col}_missing": df[col].isna().astype(int)
        for col in continuous_measurement_cols}, index=df.index)

#convert continuous measurement columns to numeric before median imputation
for col in continuous_measurement_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")
    median_value = df[col].median()
    if pd.isna(median_value):
        median_value = 0

    df[col] = df[col].fillna(median_value)

df = pd.concat([df, missing_indicator_df], axis=1) #append missingness indicators in one operation
df = df.copy() #defragment dataframe after adding many columns

print("Missing values after expanded T2.2 handling:")
missing_after_handling = df.isna().sum()
print(missing_after_handling[missing_after_handling > 0].sort_values(ascending=False))
print("\nNumber of lab value columns imputed:", len(lab_value_cols))
print("Number of vital value columns imputed:", len(vital_value_cols))
print("Number of missingness indicators created:", len([col for col in df.columns if col.endswith("_missing")]))
print("Dataset shape after missing value handling:", df.shape)


/var/folders/82/g8nb_b515zjb6xz0gjlq00mc0000gn/T/ipykernel_60007/198872629.py:62: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["has_drg_record"] = (df["drg_code_count"] > 0).astype(int)


Missing values after expanded T2.2 handling:
last_discharge_planning_order_hours        238491
first_discharge_planning_order_hours       238491
ed_to_ward_delay_hours                     234979
last_safety_order_hours_from_admission     195613
first_safety_order_hours_from_admission    195613
days_since_previous_icu_admission          189921
time_to_first_transfer_hours               186036
previous_height_cm_before_admission        160681
height_change_recent                       160681
previous_bmi_before_admission              134143
bmi_change_recent                          134143
min_previous_admission_gap_days            128201
mean_previous_admission_gap_days           128201
time_between_last_two_admissions_days      128201
std_previous_admission_gap_days            128201
weight_change_recent                       123210
previous_weight_kg_before_admission        123210
latest_height_cm_before_admission          111956
num_started_emar_events                    106633
emar_

**Check numerical variables**

Numerical variables are checked for missing or implausible values. This includes age, hospital length of stay, ICU stay count, total ICU length of stay, and the ICU indicator.

In [9]:
#check key numerical features after cleaning
lab_value_cols = [col for col in df.columns if "_lab_" in col and (col.endswith("_first_value")
        or col.endswith("_last_value") or col.endswith("_mean_value") or col.endswith("_min_value")
        or col.endswith("_max_value") or col.endswith("_std_value") or col.endswith("_value_change")
        or col.endswith("_value_range") or col.endswith("_first_24h_mean")
        or col.endswith("_first_24h_min") or col.endswith("_first_24h_max")
        or col.endswith("_first_72h_mean") or col.endswith("_first_72h_min")
        or col.endswith("_first_72h_max"))]

vital_value_cols = [col for col in df.columns if "_vital_" in col and not col.endswith("_vital_measured")]
missing_indicator_cols = [col for col in df.columns if col.endswith("_missing")]

numeric_cols = ["anchor_age", "hospital_los_days", "previous_total_admissions",
    "previous_psych_admissions_from_all_hosp", "previous_nonpsych_admissions", "icu_stay_count",
    "total_icu_los_days", "num_service_records", "num_unique_services", "num_service_transfers",
    "num_transfer_events", "num_careunit_transfers", "num_unique_careunits",
    "num_procedures", "num_unique_procedure_codes",
    "drg_severity", "drg_mortality", "drg_code_count", "num_total_diagnoses", "num_psych_diagnoses",
    "num_nonpsych_diagnoses", "num_prescription_rows", "num_unique_drugs", "num_psych_med_classes"]

numeric_cols = [col for col in numeric_cols + lab_value_cols + vital_value_cols + missing_indicator_cols
    if col in df.columns]

print("Numerical columns checked:")
print(numeric_cols)

print("\nMissing values in numerical columns:")
print(df[numeric_cols].isna().sum().sort_values(ascending=False))

print("\nSummary statistics for numerical columns:")
print(df[numeric_cols].describe().T)


Numerical columns checked:
['anchor_age', 'hospital_los_days', 'previous_total_admissions', 'previous_psych_admissions_from_all_hosp', 'previous_nonpsych_admissions', 'icu_stay_count', 'total_icu_los_days', 'num_service_records', 'num_unique_services', 'num_service_transfers', 'num_transfer_events', 'num_careunit_transfers', 'num_unique_careunits', 'num_procedures', 'num_unique_procedure_codes', 'drg_severity', 'drg_mortality', 'drg_code_count', 'num_total_diagnoses', 'num_psych_diagnoses', 'num_nonpsych_diagnoses', 'num_prescription_rows', 'num_unique_drugs', 'num_psych_med_classes', 'creatinine_lab_first_24h_max', 'glucose_lab_first_24h_max', 'hemoglobin_lab_first_24h_max', 'platelet_lab_first_24h_max', 'potassium_lab_first_24h_max', 'sodium_lab_first_24h_max', 'urea_nitrogen_lab_first_24h_max', 'wbc_lab_first_24h_max', 'creatinine_lab_first_24h_mean', 'glucose_lab_first_24h_mean', 'hemoglobin_lab_first_24h_mean', 'platelet_lab_first_24h_mean', 'potassium_lab_first_24h_mean', 'sodium

In [10]:
binary_cols = [col for col in df.columns if (col.startswith("has_") or col.startswith("had_") or
    col.endswith("_missing") or col.endswith("_measured") or col.endswith("_abnormal"))]

invalid_binary_cols = []
for col in binary_cols:
    unique_vals = sorted(df[col].dropna().unique())
    if not set(unique_vals).issubset({0, 1}):
        invalid_binary_cols.append((col, unique_vals))

if len(invalid_binary_cols) == 0:
    print("All binary indicator variables contain valid 0/1 values.")
else:
    print("Invalid binary columns detected:")
    for col, vals in invalid_binary_cols:
        print(col, vals)

All binary indicator variables contain valid 0/1 values.


**Check categorical variables after cleaning**

After missing values have been handled, categorical variables are inspected again. This helps confirm that missing values have been replaced correctly and shows the main categories present in the dataset.

In [11]:
#check categorical and binary features after cleaning
base_categorical_cols = ["gender", "anchor_year_group", "admission_type", "admission_location",
    "discharge_location", "insurance", "language", "marital_status", "race", "first_icu_careunit",
    "last_icu_careunit", "icu_type", "first_service", "last_service"]

binary_and_indicator_cols = [col for col in df.columns if col.startswith("has_")
    or col.startswith("had_") or col.endswith("_measured") or col.endswith("_abnormal")
    or col.endswith("_missing")]

categorical_cols = [col for col in base_categorical_cols + binary_and_indicator_cols if col in df.columns]
print("Categorical / indicator columns checked:")
print(categorical_cols)
print("\nMissing values in categorical / indicator columns:")
print(df[categorical_cols].isna().sum().sort_values(ascending=False))

for col in base_categorical_cols:
    if col in df.columns:
        print("\n" + "=" * 50)
        print(col)
        print("=" * 50)
        print(df[col].value_counts(dropna=False).head(15))

Categorical / indicator columns checked:
['gender', 'anchor_year_group', 'admission_type', 'admission_location', 'discharge_location', 'insurance', 'language', 'marital_status', 'race', 'first_icu_careunit', 'last_icu_careunit', 'icu_type', 'first_service', 'last_service', 'had_icu_stay', 'had_service_transfer', 'had_ed_transfer_record', 'had_unknown_transfer_careunit', 'had_procedure', 'has_anxiety_ptsd', 'has_bipolar_disorder', 'has_cognitive_delirium', 'has_depression', 'has_eating_disorder', 'has_neurodevelopmental_disorder', 'has_other_psych_diagnosis', 'has_personality_disorder', 'has_psychotic_disorder', 'has_substance_use', 'had_antidepressant', 'had_antipsychotic', 'had_mood_stabiliser', 'had_benzodiazepine', 'had_stimulant', 'had_sedative_hypnotic', 'creatinine_lab_abnormal', 'glucose_lab_abnormal', 'hemoglobin_lab_abnormal', 'platelet_lab_abnormal', 'potassium_lab_abnormal', 'sodium_lab_abnormal', 'urea_nitrogen_lab_abnormal', 'wbc_lab_abnormal', 'creatinine_lab_measured', '

**Check expanded EHR feature domains**

The cleaned dataset was reviewed by feature domain to confirm that the additional physiological, biochemical, ICU, service-transfer, physical transfer, procedure burden, DRG severity, diagnosis, and medication features from T2.1 were retained after preprocessing.

In [12]:
#summarise feature domains retained after T2.2 preprocessing
#explicitly identify diagnosis feature columns because some diagnosis flags do not contain "diagnos" in the name
diagnosis_flag_cols = ["has_anxiety_ptsd", "has_bipolar_disorder", "has_cognitive_delirium",
    "has_depression", "has_eating_disorder", "has_neurodevelopmental_disorder",
    "has_other_psych_diagnosis", "has_personality_disorder", "has_psychotic_disorder", "has_substance_use"]

diagnosis_feature_cols = [col for col in df.columns if "diagnos" in col.lower() or col in diagnosis_flag_cols]
feature_domain_summary = pd.DataFrame({"feature_domain": ["Administrative / demographic",
        "Prior admission history", "ICU utilisation and ICU type", "Service transfer features",
        "Physical transfer features", "Procedure burden features", "DRG severity and mortality",
        "Diagnosis burden and psychiatric diagnosis flags", "Medication exposure and medication burden",
        "Laboratory values and abnormality indicators", "ICU vital sign values and missingness indicators"],
    "number_of_features": [len([col for col in df.columns if col in [
            "gender", "anchor_age", "anchor_year_group", "admission_type",
            "admission_location", "discharge_location", "insurance",
            "language", "marital_status", "race", "hospital_los_days"]]),
        len([col for col in df.columns if col.startswith("previous_")]),
        len([col for col in df.columns if "icu" in col.lower()]),
        len([col for col in df.columns if "service" in col.lower()]),
        len([col for col in df.columns if col in ["num_transfer_events", "num_careunit_transfers",
            "num_unique_careunits", "had_ed_transfer_record", "had_unknown_transfer_careunit"]]),
        len([col for col in df.columns if col in ["num_procedures", "num_unique_procedure_codes", "had_procedure"]]),
        len([col for col in df.columns if "drg" in col.lower()]),
        len(diagnosis_feature_cols),
        len([col for col in df.columns if "drug" in col.lower() or "med" in col.lower() or col.startswith("had_")]),
        len([col for col in df.columns if "_lab_" in col]),
        len([col for col in df.columns if "_vital_" in col])]})

print("Expanded EHR feature domain summary:")
print(feature_domain_summary)
print("\nDataset shape after T2.2 preprocessing:")
print(df.shape)

Expanded EHR feature domain summary:
                                      feature_domain  number_of_features
0                       Administrative / demographic                  11
1                            Prior admission history                  61
2                       ICU utilisation and ICU type                  17
3                          Service transfer features                  12
4                         Physical transfer features                   5
5                          Procedure burden features                   3
6                         DRG severity and mortality                   4
7   Diagnosis burden and psychiatric diagnosis flags                  14
8          Medication exposure and medication burden                  92
9       Laboratory values and abnormality indicators                 283
10  ICU vital sign values and missingness indicators                  91

Dataset shape after T2.2 preprocessing:
(238491, 913)


**Verify outcome variable**

The readmission outcome should be binary, with 0 representing no psychiatric readmission within 30 days and 1 representing psychiatric readmission within 30 days.

In [13]:
print("Outcome values:")
print(df["readmitted_30d"].value_counts(dropna=False))

print("\nUnique outcome values:")
print(df["readmitted_30d"].unique())

Outcome values:
readmitted_30d
0    191133
1     47358
Name: count, dtype: int64

Unique outcome values:
[0 1]


**Final cleaned dataset checks**

The final cleaned dataset is checked for shape, missing values, duplicate admission records, and outcome distribution before saving. This cleaned dataset will be used for feature engineering in T2.3.

In [14]:
print("Final cleaned dataset shape:", df.shape)
print("\nDuplicate subject_id + hadm_id rows:")
print(df.duplicated(subset=["subject_id", "hadm_id"]).sum())

final_missing = pd.DataFrame({"missing_count": df.isna().sum(),
    "missing_percent": (df.isna().mean() * 100).round(2)}).sort_values("missing_percent", ascending=False)

print("\nFinal missing values summary:")
print(final_missing)

print("\nColumns with remaining missing values:")
print(final_missing[final_missing["missing_count"] > 0])

print("\nFinal readmission distribution:")
print(df["readmitted_30d"].value_counts())

print("\nFinal readmission distribution (%):")
print((df["readmitted_30d"].value_counts(normalize=True) * 100).round(2))

print("\nFinal dataset memory usage (MB):")
print(round(df.memory_usage(deep=True).sum() / 1024**2, 2))

print("\nFinal preview:")
print(df.head())

Final cleaned dataset shape: (238491, 913)

Duplicate subject_id + hadm_id rows:
0

Final missing values summary:
                                                 missing_count  \
last_discharge_planning_order_hours                     238491   
first_discharge_planning_order_hours                    238491   
ed_to_ward_delay_hours                                  234979   
last_safety_order_hours_from_admission                  195613   
first_safety_order_hours_from_admission                 195613   
...                                                        ...   
mean_bp_noninvasive_vital_value_change_missing               0   
respiratory_rate_vital_value_change_missing                  0   
spo2_vital_value_change_missing                              0   
systolic_bp_arterial_vital_value_change_missing              0   
readmitted_30d                                               0   

                                                 missing_percent  
last_discharge_planning_or

**Save cleaned dataset**

The cleaned dataset is saved for T2.3. This file keeps the admission-level cohort structure while removing data leakage and addressing missing and invalid values.

In [15]:
output_file = output_path / "t2_2_cleaned_dataset.csv"
df.to_csv(output_file, index=False)
print("Saved cleaned dataset to:")
print(output_file)

Saved cleaned dataset to:
/Users/ahthini/Desktop/DissProject/outputs/t2_2_cleaned_dataset.csv


#### **T2.2.2 - Clean long-format vital-sign time-series dataset for future sequential modelling**

**Load and clean long-format ICU vital-sign time-series data**

The long-format vital-sign dataset created in T2.1 was cleaned separately from the admission-level modelling dataset. This file preserves repeated physiological measurements over time and is intended for possible future sequential modelling, such as LSTM or transformer-based approaches. Cleaning included restricting records to admissions retained after T2.2.1, validating timestamps, removing implausible physiological values, calculating time since admission, and saving a cleaned long-format time-series dataset.

In [16]:
#load cleaned admission-level cohort from T2.2.1
cleaned_admission_file = output_path / "t2_2_cleaned_dataset.csv"
admission_reference_cols = ["subject_id", "hadm_id", "readmitted_30d"]
admission_reference = pd.read_csv(cleaned_admission_file, usecols=admission_reference_cols)
print("Cleaned admission reference shape:")
print(admission_reference.shape)
print("\nAdmission reference preview:")
print(admission_reference.head())

Cleaned admission reference shape:
(238491, 3)

Admission reference preview:
   subject_id   hadm_id  readmitted_30d
0    10000032  22595853               0
1    10000032  22841357               1
2    10000032  29079034               1
3    10000032  25742920               0
4    10000068  25022803               0


**Define plausible vital-sign ranges**

Simple adult physiological plausibility ranges were applied to remove clearly invalid charted values. These ranges are intended for data cleaning rather than clinical diagnosis.

In [17]:
#plausible physiological ranges for selected ICU vital signs
vital_plausible_ranges = {"heart_rate": (20, 250), "respiratory_rate": (4, 80),
    "spo2": (50, 100), "systolic_bp_arterial": (40, 260), "systolic_bp_noninvasive": (40, 260),
    "mean_bp_arterial": (20, 200), "mean_bp_noninvasive": (20, 200)}

print("Vital sign plausibility ranges:")
for vital_name, value_range in vital_plausible_ranges.items():
    print(vital_name, value_range)

Vital sign plausibility ranges:
heart_rate (20, 250)
respiratory_rate (4, 80)
spo2 (50, 100)
systolic_bp_arterial (40, 260)
systolic_bp_noninvasive (40, 260)
mean_bp_arterial (20, 200)
mean_bp_noninvasive (20, 200)


**Clean vital-sign time-series data in chunks**

The time-series file is large, so it was processed in chunks. Each chunk was restricted to the cleaned admission-level cohort, timestamps were converted to datetime format, implausible values were removed, and time since admission was calculated in hours.

In [18]:
#clean long-format vital-sign time-series data in chunks
timeseries_input_file = output_path / "t2_1_selected_vital_sign_timeseries.csv"
timeseries_output_file = output_path / "t2_2_cleaned_vital_sign_timeseries.csv"

chunk_size = 1_000_000
first_chunk = True

total_rows_read = 0
total_rows_after_cleaning = 0
total_rows_removed_missing = 0
total_rows_removed_time = 0
total_rows_removed_range = 0

vital_counts_before = {}
vital_counts_after = {}

print("Starting T2.2.2 vital-sign time-series cleaning...")

for chunk_number, chunk in enumerate(pd.read_csv(timeseries_input_file, chunksize=chunk_size)):
    total_rows_read += len(chunk)

    #standardise key column types
    chunk["subject_id"] = pd.to_numeric(chunk["subject_id"], errors="coerce")
    chunk["hadm_id"] = pd.to_numeric(chunk["hadm_id"], errors="coerce")
    chunk["valuenum"] = pd.to_numeric(chunk["valuenum"], errors="coerce")

    chunk["charttime"] = pd.to_datetime(chunk["charttime"], errors="coerce")
    chunk["admittime"] = pd.to_datetime(chunk["admittime"], errors="coerce")
    chunk["dischtime"] = pd.to_datetime(chunk["dischtime"], errors="coerce")

    #remove rows with missing key fields
    before_missing = len(chunk)
    chunk = chunk.dropna(subset=["subject_id", "hadm_id", "charttime",
            "admittime", "dischtime", "vital_name", "valuenum"]).copy()
    total_rows_removed_missing += before_missing - len(chunk)

    #convert identifiers back to integer after dropping missing IDs
    chunk["subject_id"] = chunk["subject_id"].astype(int)
    chunk["hadm_id"] = chunk["hadm_id"].astype(int)

    #restrict to admissions retained in the cleaned admission-level dataset
    chunk = chunk.merge(admission_reference, on=["subject_id", "hadm_id"], how="inner")

    #count vital sign rows before range filtering
    for vital_name, count in chunk["vital_name"].value_counts().items():
        vital_counts_before[vital_name] = vital_counts_before.get(vital_name, 0) + count

    #retain observations during the index admission only
    before_time = len(chunk)
    chunk = chunk[(chunk["charttime"] >= chunk["admittime"])
        & (chunk["charttime"] <= chunk["dischtime"])].copy()
    total_rows_removed_time += before_time - len(chunk)

    #calculate time since admission and time until discharge
    chunk["hours_since_admission"] = ((chunk["charttime"] - chunk["admittime"]).dt.total_seconds() / 3600)
    chunk["hours_until_discharge"] = ((chunk["dischtime"] - chunk["charttime"]).dt.total_seconds() / 3600)

    #remove implausible physiological values by vital type
    before_range = len(chunk)
    plausible_mask = pd.Series(False, index=chunk.index)
    for vital_name, (lower_bound, upper_bound) in vital_plausible_ranges.items():
        vital_mask = ((chunk["vital_name"] == vital_name) & (chunk["valuenum"] >= lower_bound)
            & (chunk["valuenum"] <= upper_bound))
        plausible_mask = plausible_mask | vital_mask

    chunk = chunk[plausible_mask].copy()
    total_rows_removed_range += before_range - len(chunk)

    #sort within chunk for chronological structure
    chunk = chunk.sort_values(["subject_id", "hadm_id", "vital_name", "charttime"])

    #count retained vital sign rows
    for vital_name, count in chunk["vital_name"].value_counts().items():
        vital_counts_after[vital_name] = vital_counts_after.get(vital_name, 0) + count

    total_rows_after_cleaning += len(chunk)

    #save cleaned chunks incrementally
    chunk.to_csv(timeseries_output_file, mode="w" if first_chunk else "a",
        header=first_chunk, index=False)

    first_chunk = False
    print("Processed chunk:", chunk_number, "| rows retained:", len(chunk))

print("\nFinished cleaning vital-sign time-series data.")
print("Rows read:", total_rows_read)
print("Rows retained:", total_rows_after_cleaning)
print("Rows removed due to missing key fields:", total_rows_removed_missing)
print("Rows removed outside admission window:", total_rows_removed_time)
print("Rows removed outside plausible value ranges:", total_rows_removed_range)

print("\nSaved cleaned time-series dataset to:")
print(timeseries_output_file)

Starting T2.2.2 vital-sign time-series cleaning...
Processed chunk: 0 | rows retained: 998466
Processed chunk: 1 | rows retained: 998646
Processed chunk: 2 | rows retained: 998829
Processed chunk: 3 | rows retained: 998583
Processed chunk: 4 | rows retained: 998808
Processed chunk: 5 | rows retained: 998740
Processed chunk: 6 | rows retained: 998704
Processed chunk: 7 | rows retained: 998698
Processed chunk: 8 | rows retained: 998656
Processed chunk: 9 | rows retained: 998872
Processed chunk: 10 | rows retained: 998626
Processed chunk: 11 | rows retained: 998806
Processed chunk: 12 | rows retained: 998789
Processed chunk: 13 | rows retained: 998841
Processed chunk: 14 | rows retained: 998784
Processed chunk: 15 | rows retained: 998783
Processed chunk: 16 | rows retained: 998692
Processed chunk: 17 | rows retained: 998671
Processed chunk: 18 | rows retained: 998773
Processed chunk: 19 | rows retained: 998752
Processed chunk: 20 | rows retained: 998801
Processed chunk: 21 | rows retained

**Review cleaned vital-sign time-series dataset**

The cleaned long-format time-series dataset was reviewed for retained vital-sign types, observation counts, time coverage, and readmission label distribution.

In [19]:
#load a sample of the cleaned time-series file for inspection
cleaned_timeseries_sample = pd.read_csv(timeseries_output_file, nrows=100000)

print("Cleaned time-series sample shape:")
print(cleaned_timeseries_sample.shape)

print("\nCleaned time-series columns:")
print(cleaned_timeseries_sample.columns.tolist())

print("\nCleaned time-series preview:")
print(cleaned_timeseries_sample.head())

print("\nVital sign counts before plausibility filtering:")
print(pd.Series(vital_counts_before).sort_values(ascending=False))

print("\nVital sign counts after plausibility filtering:")
print(pd.Series(vital_counts_after).sort_values(ascending=False))

print("\nReadmission distribution in cleaned time-series sample:")
print(cleaned_timeseries_sample["readmitted_30d"].value_counts())

print("\nTime since admission summary in cleaned time-series sample:")
print(cleaned_timeseries_sample["hours_since_admission"].describe())

print("\nNumber of admissions represented in cleaned time-series sample:")
print(cleaned_timeseries_sample["hadm_id"].nunique())

Cleaned time-series sample shape:
(100000, 12)

Cleaned time-series columns:
['subject_id', 'hadm_id', 'stay_id', 'charttime', 'itemid', 'valuenum', 'vital_name', 'admittime', 'dischtime', 'readmitted_30d', 'hours_since_admission', 'hours_until_discharge']

Cleaned time-series preview:
   subject_id   hadm_id   stay_id            charttime  itemid  valuenum  \
0    10000032  29079034  39553978  2180-07-23 14:12:00  220045      91.0   
1    10000032  29079034  39553978  2180-07-23 14:30:00  220045      93.0   
2    10000032  29079034  39553978  2180-07-23 15:00:00  220045      94.0   
3    10000032  29079034  39553978  2180-07-23 16:00:00  220045     105.0   
4    10000032  29079034  39553978  2180-07-23 17:00:00  220045      97.0   

   vital_name            admittime            dischtime  readmitted_30d  \
0  heart_rate  2180-07-23 12:35:00  2180-07-25 17:55:00               1   
1  heart_rate  2180-07-23 12:35:00  2180-07-25 17:55:00               1   
2  heart_rate  2180-07-23 12:35

In [20]:
#summarise full cleaned time-series extraction using counters collected during chunked processing
timeseries_cleaning_summary = pd.DataFrame({"check": ["Rows read from T2.1 time-series file",
        "Rows retained after cleaning", "Rows removed due to missing key fields",
        "Rows removed outside admission window", "Rows removed outside plausible ranges",
        "Percentage of rows retained", "Percentage of rows removed as implausible"],

    "value": [total_rows_read, total_rows_after_cleaning, total_rows_removed_missing,
        total_rows_removed_time, total_rows_removed_range,
        round((total_rows_after_cleaning / total_rows_read) * 100, 2),
        round((total_rows_removed_range / total_rows_read) * 100, 2)]})

print("T2.2.2 time-series cleaning audit summary:")
print(timeseries_cleaning_summary)

full_ts_patients = set()
full_ts_admissions = set()

for chunk in pd.read_csv(timeseries_output_file, usecols=["subject_id", "hadm_id"],
    chunksize=1_000_000):
    full_ts_patients.update(chunk["subject_id"].unique())
    full_ts_admissions.update(chunk["hadm_id"].unique())

print("Unique patients in cleaned time-series file:", len(full_ts_patients))
print("Unique admissions in cleaned time-series file:", len(full_ts_admissions))

T2.2.2 time-series cleaning audit summary:
                                       check        value
0       Rows read from T2.1 time-series file  21576588.00
1               Rows retained after cleaning  21549225.00
2     Rows removed due to missing key fields         0.00
3      Rows removed outside admission window         0.00
4      Rows removed outside plausible ranges     27363.00
5                Percentage of rows retained        99.87
6  Percentage of rows removed as implausible         0.13
Unique patients in cleaned time-series file: 32185
Unique admissions in cleaned time-series file: 40466
